# Voice AI: writes like you, sounds like you (100% free version)

Pipeline: your text -> Gemini (free tier) + your style guide -> XTTS-v2 cloned voice -> audio.

**Before you start**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Get a free key at aistudio.google.com/apikey (Google account, no card needed).
3. Click the key icon (Secrets) on the left, add a secret named `GEMINI_API_KEY`, paste the key, and turn on notebook access.

Colab resets when idle, so re-run these cells each session (the voice model re-downloads, about 2 GB).
Note: on Google's free tier, your prompts may be used to improve their products, so don't upload anything you consider private.

## 1. Install (about 2-3 minutes)

In [3]:
!pip install -q coqui-tts google-genai

## 2. Upload your samples
Select **all** your files in one go: the `.txt` writing samples and the `.wav` voice recordings. They're sorted into the right folders automatically.


In [4]:
import os, shutil
from google.colab import files

for d in ("samples/text", "samples/audio"):
    shutil.rmtree(d, ignore_errors=True)
    os.makedirs(d)

print("Select ALL your files at once (hold Ctrl): the .txt writing files AND the .wav voice files")
for name in files.upload():
    low = name.lower()
    if low.endswith(".txt"):
        shutil.move(name, f"samples/text/{name}")
    elif low.endswith((".wav", ".mp3", ".m4a", ".flac")):
        shutil.move(name, f"samples/audio/{name}")
    else:
        print("Skipped (not .txt or audio):", name)

print("text:", os.listdir("samples/text"))
print("audio:", os.listdir("samples/audio"))

Select ALL your files at once (hold Ctrl): the .txt writing files AND the .wav voice files


Saving my_writing_samples.txt to my_writing_samples.txt
Saving my_writing_samples_2.txt to my_writing_samples_2.txt
Saving voice_sample_1.wav to voice_sample_1.wav
Saving voice_sample_2.wav to voice_sample_2.wav
Saving voice_sample_4.wav to voice_sample_4.wav
text: ['my_writing_samples_2.txt', 'my_writing_samples.txt']
audio: ['voice_sample_2.wav', 'voice_sample_1.wav', 'voice_sample_4.wav']


## 3. Build your writing style guide

In [5]:
import glob
from pathlib import Path
from google import genai
from google.genai import types
from google.colab import userdata

try:
    api_key = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    api_key = getpass("Gemini API key: ")
client = genai.Client(api_key=api_key)

# Tried in order; if one is unavailable or out of free quota, the next is used.
MODELS = ["gemini-3.5-flash", "gemini-3.8-flash", "gemini-3.1-flash-lite"]

def generate(system, contents):
    last = None
    for m in MODELS:
        try:
            r = client.models.generate_content(
                model=m,
                contents=contents,
                config=types.GenerateContentConfig(system_instruction=system),
            )
            return r.text
        except Exception as e:
            print(f"[{m} failed: {str(e)[:120]}]")
            last = e
    raise last

samples, total = [], 0
for f in sorted(glob.glob("samples/text/*.txt")):
    t = Path(f).read_text(encoding="utf-8", errors="ignore").strip()
    if t and total + len(t) < 150_000:
        samples.append(t)
        total += len(t)
assert samples, "No .txt samples found. Re-run the upload cell."

style_guide = generate(
    "You are a stylometry expert. Analyse writing samples and produce a precise style guide.",
    "Below are writing samples from one person. Write a style guide another writer could follow "
    "to imitate them. Cover: tone, sentence length and rhythm, vocabulary and favourite phrases, "
    "punctuation and capitalisation habits, emoji use, how they open and close messages, humour, "
    "any language mixing, and what they never do. "
    "Then list 8 short verbatim example lines that best show their voice.\n\n"
    "<samples>\n" + "\n\n---\n\n".join(samples) + "\n</samples>",
)
Path("style_guide.md").write_text(style_guide, encoding="utf-8")
print(style_guide)

[gemini-3.5-flash failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand ]
[gemini-3.8-flash failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand ]
This style guide captures the essence of a writer who balances professional financial discipline with a relaxed, vernacular Kenyan social persona. Their voice shifts distinctly between the "Work Professional" (direct, structured, urgent) and the "Mtaa Persona" (colloquial, sarcastic, emoji-heavy).

### **Style Guide**

**1. Tone**
*   **Professional/Client-Facing:** Extremely transactional, urgent, and focused on deadlines and documentation. Often authoritative ("You are advised to...").
*   **Social/Peer-Facing:** Informal, slang-heavy, slightly provocative, and prone to "banter." They treat peers like family, using affection and ridicule interchangeably.

**2. Sentence Length and Rhythm**
*   

## 4. Load the voice model
Running this cell means you accept Coqui's XTTS-v2 licence (non-commercial use). XTTS-v2 does not support Swahili.

In [7]:
import os, glob, torch
os.environ["COQUI_TOS_AGREED"] = "1"
from TTS.api import TTS
from IPython.display import Audio, display

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device, "(switch the runtime to a T4 GPU if this says cpu)")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)

ref_clips = sorted(
    p for ext in ("*.wav", "*.mp3", "*.m4a", "*.flac") for p in glob.glob(f"samples/audio/{ext}")
)[:3]
assert ref_clips, "No voice samples found. Re-run the upload cell."

def speak(text, path="reply.wav"):
    tts.tts_to_file(text=text, speaker_wav=ref_clips, language="en", file_path=path)
    display(Audio(path, autoplay=True))

speak("Hello, this is a test of my cloned voice.")

Device: cuda (switch the runtime to a T4 GPU if this says cpu)


In [6]:
!pip install -q "transformers==4.57.6"

## 5. Chat in your voice (type `quit` to stop)

In [ ]:
persona = (
    "You are a digital twin of the person whose style guide follows. Reply exactly as they would: "
    "same tone, rhythm, vocabulary and habits. Write for speech: no markdown, no lists, no emoji "
    "unless the guide says they use them, and keep replies short (1-3 sentences) and conversational. "
    "Never claim to be a different person, and if asked, admit you are an AI imitating their style.\n\n"
    f"<style_guide>\n{style_guide}\n</style_guide>"
)

history = []

def turn(role, text):
    return types.Content(role=role, parts=[types.Part(text=text)])

while True:
    user = input("\nYou: ").strip()
    if user.lower() in ("quit", "exit"):
        break
    if not user:
        continue
    history.append(turn("user", user))
    reply = generate(persona, history)
    history.append(turn("model", reply))
    print("AI:", reply)
    speak(reply)


You: sir, kindly share copy of logbook
[gemini-3.5-flash failed: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand ]
AI: Good day. Scanned copy sent to your email. Kindly confirm receipt and expedite the documentation.


## 6. Save your style guide (so you can skip step 3 next time)

In [ ]:
from google.colab import files
files.download("style_guide.md")

In [ ]:
import os, json
from google.colab import files
D = os.environ['VT_DATA']
open('voice-style.md','w').write(open(D+'/styles/current.md').read())
ex = [{"client": o["prompt"], "reply": o["ideal"]}
         for l in open(D+'/feedback.jsonl') for o in [json.loads(l)]]
json.dump(ex, open('voice-examples.json','w'))
files.download('voice-style.md'); files.download('voice-examples.json')